In [2]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class ModeAndFeatureDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f['OwnSpeed'] / 3.6 for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f['TgtSpeed_ref'] / 3.6 for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)  # 🔽 相対加速度を追加

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed = np.mean(t - s)

                feature = np.concatenate([modes, d, s, a, s1, d1, d2, rel_acc] + d_smooths)
                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid

# -------- Collate --------
def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)

# -------- Model --------
class ExtendedFeatureModel(nn.Module):
    def __init__(self, in_dim=225):  # 必要に応じて input 次元を調整
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x).squeeze(1)

# -------- Training Loop --------
def train_extended_model(dataset, save_path="420_2.pth"):
    scenes = sorted(list(set([item[-1] for item in dataset.items])))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)

    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx[:8000])
    val_ds = Subset(dataset, val_idx[:2000])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ExtendedFeatureModel(in_dim=train_ds[0][0].shape[0]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-2)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 250
    patience_counter = 0

    for epoch in range(250):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path} (val_loss={val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\u23f9 Early stopping at epoch {epoch+1}")
                break

    return model


In [9]:
# --- データセットとモデルの読み込み・実行 ---
crop_root = "../train/disparity_crops"
annot_root = "../train/train_annotations"
distance_json_path = "../distance_estimates_filtered.json"

# データセット作成
dataset = ModeAndFeatureDataset(
    crop_root=crop_root,
    annot_root=annot_root,
    distance_json_path=distance_json_path,
    max_items=10000
)

# 学習実行
model = train_extended_model(dataset, save_path="420_2.pth")


[Train 1]: 100%|██████████| 125/125 [00:01<00:00, 103.46it/s]


Epoch 1 | Train Loss: 0.4037 | Val Loss: 0.1885
Model saved to 420_2.pth (val_loss=0.1885)


[Train 2]: 100%|██████████| 125/125 [00:00<00:00, 253.59it/s]


Epoch 2 | Train Loss: 0.3309 | Val Loss: 0.2152


[Train 3]: 100%|██████████| 125/125 [00:00<00:00, 250.63it/s]


...

Epoch 248 | Train Loss: 0.1594 | Val Loss: 0.1938


[Train 249]: 100%|██████████| 125/125 [00:00<00:00, 250.14it/s]


Epoch 249 | Train Loss: 0.1623 | Val Loss: 0.2404


[Train 250]: 100%|██████████| 125/125 [00:00<00:00, 248.22it/s]


Epoch 250 | Train Loss: 0.1678 | Val Loss: 0.2639


In [15]:
import os
import json
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from PIL import Image
from collections import OrderedDict


CROP_ROOT = "../test/test_crops"
ANNOT_ROOT = "../test/test_annotations"
DISTANCE_JSON = "../testdistance_estimates_filtered.json"
OUTPUT_JSON = "submission_tgtspeed.json"
MODEL_PATH = "420_2.pth"


def collate_fn(batch):
    feats, tgts, sids = zip(*batch)
    return torch.stack(feats), torch.tensor(tgts), list(sids)


class ExtendedFeatureModel(torch.nn.Module):
    def __init__(self, in_dim=225):
        super().__init__()
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(in_dim, 512), torch.nn.BatchNorm1d(512), torch.nn.ReLU(),
            torch.nn.Linear(512, 256), torch.nn.BatchNorm1d(256), torch.nn.ReLU(),
            torch.nn.Linear(256, 128), torch.nn.BatchNorm1d(128), torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 64), torch.nn.BatchNorm1d(64), torch.nn.ReLU(),
            torch.nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x).squeeze(1)


class ModeAndFeatureDataset(torch.utils.data.Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=None):
        self.items = []
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if not sid.isdigit():
                continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue

            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann['sequence']
            min_len = min(len(files), len(seq))
            if min_len < 15: continue

            own_speeds = np.array([f['OwnSpeed'] / 3.6 for f in seq], dtype=np.float32)
            tgt_speeds = np.array([f.get('TgtSpeed_ref', 0.0) / 3.6 for f in seq], dtype=np.float32)
            angles = np.array([f['StrDeg'] for f in seq], dtype=np.float32)
            dists_all = [self.distances.get(sid, {}).get(str(i), 0.0) for i in range(min_len)]
            dists_all = np.array(dists_all, dtype=np.float32)

            def smooth(x, k):
                return np.convolve(x, np.ones(k)/k, mode='same')

            for i in range(min_len - 14):
                if max_items and len(self.items) >= max_items:
                    return

                d = dists_all[i:i+15]
                s = own_speeds[i:i+15]
                a = angles[i:i+15]
                t = tgt_speeds[i:i+15]

                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                s1 = np.gradient(s)
                rel_acc = d2 - np.mean(d2)

                d_smooths = []
                for w in [3, 5, 7, 11]:
                    smoothed = smooth(d, w)[:15]
                    d_smooths.append(smoothed)
                    d_smooths.append(np.gradient(smoothed))

                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                modes = []
                for p in img_paths:
                    img = np.array(Image.open(p).convert("L")).flatten()
                    if img.size == 0:
                        img_mode = 0.0
                    else:
                        vals, counts = np.unique(img, return_counts=True)
                        img_mode = float(vals[np.argmax(counts)]) / 255.0
                    modes.append(img_mode)

                rel_speed = np.mean(t - s)
                feature = np.concatenate([modes, d, s, a, s1, d1, d2, rel_acc] + d_smooths)
                self.items.append((feature.astype(np.float32), rel_speed, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feature, tgt, sid = self.items[idx]
        return torch.tensor(feature), torch.tensor(tgt, dtype=torch.float32), sid


def inference_submission(model_path, crop_root, annot_root, distance_json_path, output_json):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = ModeAndFeatureDataset(
        crop_root=crop_root,
        annot_root=annot_root,
        distance_json_path=distance_json_path,
        max_items=None
    )

    loader = DataLoader(dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

    model = ExtendedFeatureModel(in_dim=dataset[0][0].shape[0])
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    raw_predictions = OrderedDict()

    with torch.no_grad():
        for feats, _, sids in tqdm(loader, desc="Running inference"):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()

            for sid, rel_v, feat in zip(sids, preds, feats.cpu()):
                sid = str(sid)
                own_speed = np.mean(feat[30:45].numpy())  # 15個のOwnSpeed成分
                tgt_speed_kmph = float((rel_v + own_speed) * 3.6)  # m/s → km/h
                if sid not in raw_predictions:
                    raw_predictions[sid] = []
                raw_predictions[sid].append(round(tgt_speed_kmph, 6))

    # 各シーンのフレーム長を取得
    scene_lengths = OrderedDict()
    for filename in sorted(os.listdir(annot_root), key=lambda x: int(x.replace(".json", ""))):
        if filename.endswith(".json"):
            sid = filename[:-5]
            with open(os.path.join(annot_root, filename), encoding='utf-8') as f:
                ann = json.load(f)
                scene_lengths[sid] = len(ann['sequence'])

    # 推論: 先頭19フレームを0.0にし、それ以降を推論結果で補完
    final_predictions = OrderedDict()
    for sid in scene_lengths:
        pred = raw_predictions.get(sid, [])
        target_len = scene_lengths[sid]

        padded_pred = [0.0] * 19 + pred

        if len(padded_pred) < target_len:
            pad_value = padded_pred[-1] if padded_pred else 0.0
            padded_pred += [pad_value] * (target_len - len(padded_pred))

        final_predictions[sid] = padded_pred[:target_len]

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(final_predictions, f, indent=2, ensure_ascii=False)

    print(f"✅ 推論完了: {output_json} に保存しました")


# 実行
inference_submission(
    model_path=MODEL_PATH,
    crop_root=CROP_ROOT,
    annot_root=ANNOT_ROOT,
    distance_json_path=DISTANCE_JSON,
    output_json=OUTPUT_JSON
)


Running inference: 100%|██████████| 379/379 [00:01<00:00, 302.03it/s]


✅ 推論完了: submission_tgtspeed.json に保存しました
